# Example 8.2 / 1: Read a Solver Log

Companion notebook for **8.2 Solver Options and Logs** in the AMPL lab Documentation, section "Read a Solver Log". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Read a Solver Log

The HiGHS option `tech:outlev=1` prints the solver's log. This example solves the steel model
with it. In `data.py` and `run.py`, `ampl` is an `AMPL` object that has already read
`model.mod`:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;                                 # products
param rate {PROD} > 0;                    # tons produced per hour
param avail >= 0;                         # hours available
param profit {PROD};                      # profit per ton
param market {PROD} >= 0;                 # most tons that can be sold
var Make {p in PROD} >= 0, <= market[p];  # tons produced, up to the market limit
maximize Total_Profit: sum {p in PROD} profit[p] * Make[p];  # total profit
# production hours fit the hours available
subject to Time: sum {p in PROD} (1 / rate[p]) * Make[p] <= avail;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PROD"] = ["bands", "coils", "plate"]
ampl.param["rate"] = {"bands": 200, "coils": 140, "plate": 160}
ampl.param["profit"] = {"bands": 25, "coils": 30, "plate": 29}
ampl.param["market"] = {"bands": 6000, "coils": 4000, "plate": 3500}
ampl.param["avail"] = 40

In [5]:
# run.py
ampl.solve(solver="highs", highs_options="tech:outlev=1")
print(ampl.solve_result)

HiGHS 1.14.0:   tech:outlev = 1

AMPL MP initial flat model has 3 variables (0 integer, 0 binary);
Objectives: 1 linear; 
Constraints:  1 linear;

AMPL MP did not modify the model.


Running HiGHS 1.14.0 (git hash: 7df0786): Copyright (c) 2026 under Apache 2.0 license terms
Using BLAS: OpenBLAS 
LP has 1 row; 3 cols; 3 nonzeros
Coefficient ranges:
  Matrix  [5e-03, 7e-03]
  Cost    [2e+01, 3e+01]
  Bound   [4e+03, 6e+03]
  RHS     [4e+01, 4e+01]
Presolving model
0 rows, 0 cols, 0 nonzeros 0s
0 rows, 0 cols, 0 nonzeros 0s
Presolve reductions: rows 0(-1); columns 0(-3); nonzeros 0(-3) - Reduced to empty
Performed postsolve
Solving the original LP from the solution after postsolve

Model status        : Optimal
Objective value     :  1.9640000000e+05
P-D objective error :  0.0000000000e+00
HiGHS run time      :          0.00
HiGHS 1.14.0: optimal solution; objective 196400
0 simplex iterations
0 barrier iterations
solved


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set PROD := bands coils plate;
param rate := bands 200  coils 140  plate 160;
param profit := bands 25  coils 30  plate 29;
param market := bands 6000  coils 4000  plate 3500;
param avail := 40;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
option highs_options 'tech:outlev=1';
solve;
display solve_result;

HiGHS 1.14.0:   tech:outlev = 1

AMPL MP initial flat model has 3 variables (0 integer, 0 binary);
Objectives: 1 linear; 
Constraints:  1 linear;

AMPL MP did not modify the model.


Running HiGHS 1.14.0 (git hash: 7df0786): Copyright (c) 2026 under Apache 2.0 license terms
Using BLAS: OpenBLAS 
LP has 1 row; 3 cols; 3 nonzeros
Coefficient ranges:
  Matrix  [5e-03, 7e-03]
  Cost    [2e+01, 3e+01]
  Bound   [4e+03, 6e+03]
  RHS     [4e+01, 4e+01]
Presolving model
0 rows, 0 cols, 0 nonzeros 0s
0 rows, 0 cols, 0 nonzeros 0s
Presolve reductions: rows 0(-1); columns 0(-3); nonzeros 0(-3) - Reduced to empty
Performed postsolve
Solving the original LP from the solution after postsolve

Model status        : Optimal
Objective value     :  1.9640000000e+05
P-D objective error :  0.0000000000e+00
HiGHS run time      :          0.00
HiGHS 1.14.0: optimal solution; objective 196400
0 simplex iterations
0 barrier iterations
solve_result = solved



### Another way in amplpy

Read the log from top to bottom:

| Log lines | What they tell you |
|---|---|
| `HiGHS 1.14.0:   tech:outlev = 1` | The driver echoes every option it accepted, so a typo shows up here or as an error |
| `AMPL MP initial flat model has 3 variables ...`, `AMPL MP did not modify the model.` | The MP layer received 3 variables, 1 objective and 1 constraint and reformulated nothing (How MP Solvers Reformulate Your Model (AMPL lab documentation, section 8.6)) |
| `LP has 1 row; 3 cols; 3 nonzeros` and `Coefficient ranges` | The size and the spread of magnitudes the solver sees (Numerical Accuracy (AMPL lab documentation, section 8.7)) |
| `Presolving model` ... `Reduced to empty` | HiGHS's own presolve solved this tiny LP completely (Presolve (AMPL lab documentation, section 8.3) covers AMPL's presolve, a separate stage) |
| `Model status : Optimal`, `Objective value` | The solver's verdict, before AMPL reports it |
| `HiGHS 1.14.0: optimal solution; objective 196400` and the iteration counts | The solve message that AMPL keeps in `solve_message`; `ampl.solve_result` is `solved` |

Larger models add progress lines: simplex or barrier iterations for an LP, and incumbent, bound
and gap rows for a MIP (How MIP Solvers Work (AMPL lab documentation, section 8.9.1)). Increase verbosity only as needed.

A name the driver does not know stops the solve. With the same `model.mod` and `data.py`, an
option string copied from another solver fails:

In [9]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [10]:
# data.py
ampl.set["PROD"] = ["bands", "coils", "plate"]
ampl.param["rate"] = {"bands": 200, "coils": 140, "plate": 160}
ampl.param["profit"] = {"bands": 25, "coils": 30, "plate": 29}
ampl.param["market"] = {"bands": 6000, "coils": 4000, "plate": 3500}
ampl.param["avail"] = 40

In [11]:
# run.py
ampl.solve(solver="highs", highs_options="timelimit=60 mipgapx=0.01")
print(ampl.solve_result)

HiGHS 1.14.0:   lim:time = 60
HiGHS 1.14.0: Unknown option or invalid key "mipgapx"
failure
